# 用 Plotly 可视化流数据（Visualizing Streaming Data with Plotly）

> 运行前提：
> - 需先在另一个终端启动 tick 数据服务器（端口 5555），且与 本Notebook 在同一台机器上。
> - 第 5 节的柱状图还需要启动`BarsServer.py`（端口 5556）。

## 1. 概述与环境准备

实时可视化流数据通常要求较高。Plotly 可同时服务静态图与流式交互图。

In [12]:
import plotly
import ipywidgets

print('plotly', plotly.__version__)
print('ipywidgets', ipywidgets.__version__)

# 缺包时任选其一取消注释：
# !pip install -q plotly ipywidgets
# !conda install -y plotly ipywidgets

# 可选：仅当 JupyterLab 很旧、FigureWidget 不渲染时再开（新版一般不需要）
# !jupyter labextension install jupyterlab-plotly
# !jupyter labextension install @jupyter-widgets/jupyterlab-manager
# !jupyter labextension install plotlywidget


plotly 6.9.0
ipywidgets 8.1.8


## 2. 基础：一条实时数据流

安装好所需的包和扩展后，生成流式图表就相当高效了。

#### 2.1 第一步：创建 Plotly 图形控件（figure widget）。


In [13]:
import zmq
from datetime import datetime
import plotly.graph_objects as go  # 从 plotly 导入图形对象（graphical objects）

In [14]:
symbol = 'SYMBOL'

In [4]:
fig = go.FigureWidget()  # 在 Jupyter 中实例化 Plotly 图形控件
fig.add_scatter()        # 添加一个散点（scatter）对象
fig

FigureWidget({
    'data': [{'type': 'scatter', 'uid': '988253d5-f131-4eca-807d-f4ad57a7bd01'}], 'layout': {'template': '...'}
})

#### 2.2 第二步：建立与 tick 数据服务器的套接字通信。 

服务器需要在同一台机器上的另一个 Python 进程中运行。收到的数据会加上时间戳并收集到 `list` 对象中，这些列表再用来更新图形控件的数据对象（见书中图 7-1）。


In [15]:
context = zmq.Context()

In [6]:
socket = context.socket(zmq.SUB)

In [7]:
socket.connect('tcp://0.0.0.0:5555')

<SocketContext(connect='tcp://0.0.0.0:5555')>

In [8]:
socket.setsockopt_string(zmq.SUBSCRIBE, 'SYMBOL')

In [9]:
times = list()   # 存放时间戳
prices = list()  # 存放实时价格

In [10]:
for _ in range(50):
    msg = socket.recv_string()
    t = datetime.now()  # 生成时间戳
    times.append(t)     # 追加到 times
    # TickServer 消息形如: SYMBOL 2026-10-01 12:06:32.931 100.00
    price = float(msg.split()[-1])  # 价格在最后一段
    prices.append(price)
    fig.data[0].x = times   # 用时间更新散点 x
    fig.data[0].y = prices  # 用价格更新散点 y（图 7-1：实时价格流）

## 3. 三条实时流

一个 Plotly 流式图可以有多个图形对象。例如，除了价格 tick 外，还想实时显示两条 **简单移动平均线（SMA）** 时，这就很方便。

下面的代码再次实例化一个图形控件，这次包含三个散点对象。来自 tick 数据服务器的数据被收集到一个 pandas `DataFrame` 中，每次从套接字更新后重新计算两条 SMA，再用修改后的数据集更新图形控件的数据对象（见书中图 7-2）。


In [11]:
fig = go.FigureWidget()
fig.add_scatter(name='SYMBOL')  # 价格
fig.add_scatter(name='SMA1', line=dict(width=1, dash='dot'),
                mode='lines+markers')  # SMA1
fig.add_scatter(name='SMA2', line=dict(width=1, dash='dash'),
                mode='lines+markers')  # SMA2
fig

FigureWidget({
    'data': [{'name': 'SYMBOL', 'type': 'scatter', 'uid': '27a8afbf-418e-40f6-bb87-3285a7f9b6b3'},
             {'line': {'dash': 'dot', 'width': 1},
              'mode': 'lines+markers',
              'name': 'SMA1',
              'type': 'scatter',
              'uid': 'ee36b350-21f0-42a8-b17d-d1f53336ef81'},
             {'line': {'dash': 'dash', 'width': 1},
              'mode': 'lines+markers',
              'name': 'SMA2',
              'type': 'scatter',
              'uid': '0c90876b-a15b-44eb-b545-5d3c72565369'}],
    'layout': {'template': '...'}
})

In [12]:
import pandas as pd

In [13]:
df = pd.DataFrame()

In [14]:
for _ in range(75):
    msg = socket.recv_string()
    t = datetime.now()
    parts = msg.split()  # SYMBOL 日期 时间 价格
    symbol, price = parts[0], float(parts[-1])
    # 收集 tick 到 DataFrame（pandas 2.0+ 用 concat）
    df = pd.concat([df, pd.DataFrame({symbol: price}, index=[t])])
    df['SMA1'] = df[symbol].rolling(5).mean()   # 窗口 5
    df['SMA2'] = df[symbol].rolling(10).mean()  # 窗口 10
    fig.data[0].x = df.index
    fig.data[1].x = df.index
    fig.data[2].x = df.index
    fig.data[0].y = df[symbol]
    fig.data[1].y = df['SMA1']
    fig.data[2].y = df['SMA2']  # 图 7-2：价格 + 两条 SMA
# 练习：可把 SMA 流式图与在线交易算法结合，并加 resampling（按 bar 而非 tick）


## 4. 三个子图，三条数据流

和常规 Plotly 图一样，基于图形控件的流式图也可以有多个子*。下面的例子创建了一个含三个子图的流式图：

1. 第一个子图：实时 tick 数据；
2. 第二个子图：对数收益率（log returns）；
3. 第三个子图：基于对数收益率的**时间序列动量**。

书中图 7-3 展示了整个图形对象的一个快照。


In [15]:
from plotly.subplots import make_subplots

In [16]:
f = make_subplots(rows=3, cols=1, shared_xaxes=True)  # 三个共享 x 轴的子图
f.append_trace(go.Scatter(name='SYMBOL'), 
               row=1, col=1)  # 子图1：价格
f.append_trace(go.Scatter(name='RETURN', line=dict(width=1, dash='dot'), mode='lines+markers', marker={'symbol': 'triangle-up'}),
               row=2, col=1)  # 子图2：对数收益率
f.append_trace(go.Scatter(name='MOMENTUM', line=dict(width=1, dash='dash'), mode='lines+markers', marker={'symbol': 'x'}),
               row=3, col=1)  # 子图3：动量（图 7-3）

_ = f.update_layout(height=600)  # 可选：调整图形高度


In [17]:
fig = go.FigureWidget(f)

In [18]:
fig

FigureWidget({
    'data': [{'name': 'SYMBOL',
              'type': 'scatter',
              'uid': 'efba4ed9-71ac-4049-ba80-14572124da20',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'dash': 'dot', 'width': 1},
              'marker': {'symbol': 'triangle-up'},
              'mode': 'lines+markers',
              'name': 'RETURN',
              'type': 'scatter',
              'uid': '7b653775-583a-467b-8ff4-e121ccce4d9b',
              'xaxis': 'x2',
              'yaxis': 'y2'},
             {'line': {'dash': 'dash', 'width': 1},
              'marker': {'symbol': 'x'},
              'mode': 'lines+markers',
              'name': 'MOMENTUM',
              'type': 'scatter',
              'uid': '05e9fec0-c0cd-4277-a889-0f1bfcebaa15',
              'xaxis': 'x3',
              'yaxis': 'y3'}],
    'layout': {'height': 600,
               'template': '...',
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'matches': 'x3', 'showticklabe

In [19]:
import numpy as np

In [20]:
df = pd.DataFrame()

In [21]:
for _ in range(75):
    msg = socket.recv_string()
    t = datetime.now()
    parts = msg.split()  # SYMBOL 日期 时间 价格
    symbol, price = parts[0], float(parts[-1])
    df = pd.concat([df, pd.DataFrame({symbol: price}, index=[t])])  # 收集 tick
    df['RET'] = np.log(df[symbol] / df[symbol].shift(1))  # 对数收益率
    df['MOM'] = df['RET'].rolling(10).mean()         # 收益率动量
    fig.data[0].x = df.index
    fig.data[1].x = df.index
    fig.data[2].x = df.index
    fig.data[0].y = df[symbol]
    fig.data[1].y = df['RET']
    fig.data[2].y = df['MOM']


## 5. 以柱状图呈现流数据

并非所有流数据都最适合画成时间序列（`Scatter` 对象），有些更适合画成高度不断变化的柱状图。附录 B 提供了一个脚本，用于提供适合柱状图可视化的示例数据。一组数据（一条消息）由 8 个浮点数组成。

下面的 Python 代码生成流式柱状图（见书中图 7-4）。这种情况下，x 数据通常不变。

> 运行前提：*以下代码要工作，需要在另一个本地 Python 实例中先执行 `BarsServer.py` 脚本。


In [16]:
socket = context.socket(zmq.SUB)

In [17]:
socket.connect('tcp://0.0.0.0:5556')  # 柱状图数据端口 5556（不再是前面的 5555）


<SocketContext(connect='tcp://0.0.0.0:5556')>

In [18]:
socket.setsockopt_string(zmq.SUBSCRIBE, '')

In [19]:
for _ in range(5):
    msg = socket.recv_string()
    print(msg)

87.903 4.380 87.628 20.323 95.801 45.231 93.276 22.996
32.021 56.049 63.915 50.514 85.997 51.874 14.841 5.061
29.743 23.031 47.735 47.055 97.308 63.156 20.732 64.793
37.371 17.165 46.844 12.641 53.462 54.628 23.805 54.896
60.018 84.305 70.600 53.692 1.064 25.996 52.219 35.521


In [20]:
fig = go.FigureWidget()
fig.add_bar()  # 柱状图对象
fig


FigureWidget({
    'data': [{'type': 'bar', 'uid': 'c015a180-8d7b-46bd-8fb0-7f15b59aaf04'}], 'layout': {'template': '...'}
})

In [21]:
x = list('abcdefgh')  # x 轴通常不变
fig.data[0].x = x
for _ in range(25):
    msg = socket.recv_string()
    y = msg.split()
    y = [float(n) for n in y]
    fig.data[0].y = y  # 更新柱高（图 7-4：流式柱状图）
